# ARIMA Model — Experiment Notebook
**Task 3: Time Series Forecasting (ARIMA/SARIMA)**

This notebook:
1. Loads preprocessed data for all stocks
2. Fits auto_arima with AIC/BIC order selection
3. Validates residuals (Ljung-Box)
4. Evaluates on test set (Jul–Dec 2025)
5. Forecasts next 5 trading days with confidence intervals
6. Computes RMSE, MAPE, Directional Accuracy

In [ ]:
import sys
sys.path.append("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pmdarima as pm
from statsmodels.stats.diagnostic import acorr_ljungbox

from src.data.fetch_data import load_all_raw, STOCK_UNIVERSE
from src.data.preprocess import preprocess_all
from src.utils.metrics import evaluate

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
print("Imports OK")

## 1. Load & Preprocess Data

In [ ]:
raw = load_all_raw()
processed = preprocess_all(raw, save=False)

print(f"Loaded {len(processed)} stocks")
for ticker, d in processed.items():
    print(f"  {ticker}: train={len(d['train'])}  test={len(d['test'])}  d_order={d['d_order']}")

## 2. Single-Stock Deep Dive (TCS.NS)
Let's first understand the full ARIMA workflow on one stock before looping over all.

In [ ]:
TICKER = "TCS.NS"
train = processed[TICKER]["train"]
test  = processed[TICKER]["test"]

print(f"Train: {train.index[0].date()} → {train.index[-1].date()} ({len(train)} obs)")
print(f"Test:  {test.index[0].date()} → {test.index[-1].date()} ({len(test)} obs)")

fig, ax = plt.subplots()
ax.plot(train[-200:], label="Train (last 200d)", color="steelblue")
ax.plot(test, label="Test", color="green")
ax.axvline(test.index[0], color="red", linestyle="--", alpha=0.7, label="Split")
ax.set_title(f"{TICKER} — Train/Test Split")
ax.legend()
plt.tight_layout()
plt.show()

### 2a. Auto ARIMA — AIC/BIC Order Selection

In [ ]:
model = pm.auto_arima(
    train,
    seasonal=False,
    information_criterion="aic",
    max_p=5, max_q=5,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
)

print(f"Best order: {model.order}")
print(f"AIC: {model.aic():.2f}")
print(f"BIC: {model.bic():.2f}")
print(model.summary())

### 2b. Residual Diagnostics

In [ ]:
resid = pd.Series(model.resid())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Residual plot
axes[0].plot(resid, color="steelblue", alpha=0.7)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals")

# Histogram
axes[1].hist(resid, bins=40, color="steelblue", edgecolor="white")
axes[1].set_title("Residual Distribution")

# ACF of residuals
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(resid, lags=30, ax=axes[2], color="steelblue")
axes[2].set_title("ACF of Residuals")

plt.tight_layout()
plt.show()

# Ljung-Box test
lb = acorr_ljungbox(resid, lags=[10, 20], return_df=True)
print("\nLjung-Box Test:")
print(lb)
print(f"\nResiduals are white noise: {bool(lb['lb_pvalue'].min() > 0.05)}")

### 2c. Walk-Forward Test Predictions

In [ ]:
# Walk-forward 1-step prediction
predictions = []
m = model.copy()
for i in range(len(test)):
    fc = m.predict(n_periods=1)
    predictions.append(float(fc[0]))
    m.update([test.iloc[i]])

pred = pd.Series(predictions, index=test.index, name="ARIMA_Pred")

# Evaluate
metrics = evaluate(test.values, pred.values, "ARIMA", TICKER)
print(f"\nMetrics for {TICKER}:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Plot forecast vs actual
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train[-100:], label="Train (last 100d)", color="steelblue", alpha=0.5)
ax.plot(test, label="Actual", color="green", linewidth=2)
ax.plot(pred, label="ARIMA Forecast", color="red", linewidth=1.5, linestyle="--")
ax.set_title(f"{TICKER} — ARIMA Forecast vs Actual (Test Period)")
ax.legend()
plt.tight_layout()
plt.show()

### 2d. Future Forecast (Next 5 Trading Days) with Confidence Intervals

In [ ]:
# Re-fit on full data (train + test)
full = pd.concat([train, test])
model_full = pm.auto_arima(
    full, seasonal=False, stepwise=True,
    suppress_warnings=True, error_action="ignore"
)

N_FORECAST = 5
fc_vals, conf_int = model_full.predict(n_periods=N_FORECAST, return_conf_int=True)

# Create future dates (business days)
last_date = full.index[-1]
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=N_FORECAST)

print(f"\nForecast for {TICKER} — Next {N_FORECAST} Trading Days:")
for i, (d, v, lo, hi) in enumerate(zip(future_dates, fc_vals, conf_int[:, 0], conf_int[:, 1])):
    print(f"  {d.date()}: ₹{v:.2f}  [₹{lo:.2f} — ₹{hi:.2f}]")

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(full[-60:], label="Historical", color="steelblue")
ax.plot(future_dates, fc_vals, label="Forecast", color="red", marker="o")
ax.fill_between(future_dates, conf_int[:, 0], conf_int[:, 1],
                color="red", alpha=0.15, label="95% CI")
ax.set_title(f"{TICKER} — ARIMA 5-Day Ahead Forecast")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Run ARIMA on All Stocks

In [ ]:
from src.models.arima import run_arima_pipeline

all_preds, all_fc, all_metrics = run_arima_pipeline(processed, n_forecast=5)

metrics_df = pd.DataFrame(all_metrics)
print("\n── ARIMA Results Across All Stocks ──")
print(metrics_df.to_string())
print(f"\nAvg MAPE: {metrics_df['MAPE'].mean():.2f}%")
print(f"Avg RMSE: {metrics_df['RMSE'].mean():.2f}")
print(f"Avg Dir Accuracy: {metrics_df['DirAcc'].mean():.1f}%")

In [ ]:
# Visualise all forecasts
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (ticker, pred) in enumerate(all_preds.items()):
    if i >= 9:
        break
    ax = axes[i]
    actual = processed[ticker]["test"]
    ax.plot(actual, label="Actual", color="green", linewidth=1.5)
    ax.plot(pred, label="ARIMA", color="red", linewidth=1, linestyle="--")
    ax.set_title(f"{STOCK_UNIVERSE.get(ticker, ticker)}", fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.legend(fontsize=7)

plt.suptitle("ARIMA Forecast vs Actual — All Stocks", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Summary
- Auto ARIMA selected optimal (p,d,q) via AIC minimisation
- Residual diagnostics confirm white noise (Ljung-Box p > 0.05)
- Walk-forward evaluation gives realistic out-of-sample metrics
- Confidence intervals provided for future forecasts
- Results saved to `results/forecasts/`